In [ ]:
pip install dropbox_sign

In [ ]:
pip install google-cloud-secret-manager

In [ ]:
from google.colab import auth
from google.cloud import secretmanager
client=secretmanager.SecretManagerServiceClient()
secret_name="dropbox_sign_api"
secret_name2="dropbox_sign_account_id"
project_id="1234567890"
resource_name=f"projects/{project_id}/secrets/{secret_name}/versions/1"
resource_name2=f"projects/{project_id}/secrets/{secret_name2}/versions/1"
response=client.access_secret_version(request={"name":resource_name})
response2=client.access_secret_version(request={"name":resource_name2})
ds_api = response.payload.data.decode("UTF-8")
ds_acct = response2.payload.data.decode("UTF-8")


In [ ]:
from dropbox_sign import ApiClient, ApiException, Configuration, apis
import pandas as pd
import pandas_gbq

def normalize_all_signature_requests(api_key, account_id):
    """
    Retrieves all signature requests from Dropbox Sign across multiple pages and normalizes the JSON response.

    Args:
        api_key (str):  Dropbox Sign API key.
        account_id (str): The account ID.

    Returns:
        pandas.DataFrame or None: A DataFrame representing all signature requests,
                                    or None if an error occurs.
    """
    configuration = Configuration(
        username= ds_api 
    )
    with ApiClient(configuration) as api_client:
        try:
            all_normalized_data = []
            page = 1
            while True:
                response = apis.SignatureRequestApi(api_client).signature_request_list(
                    account_id=account_id, page=page
                )
                # Added page parameter

                if not response or not response.signature_requests:
                    break  # No more pages

                for request in response.signature_requests:
                    request_data = {
                        "title": request.title,
                        "signature_request_id" : request.signature_request_id,
                        "is_complete": request.is_complete,  # set default value.
                        "template": request.template_ids,  
                    }

                    if request.signatures:
                        for i, signature in enumerate(request.signatures):
                            request_data[f"signature_{i + 1}_signer_email_address"] = signature.signer_email_address
                            request_data[f"signature_{i + 1}_signer_name"] = signature.signer_name
                            request_data[f"signature_{i + 1}_signed_at"] = signature.signed_at
                    if request.response_data:
                        for field in request.response_data:
                            request_data[f"response_data_{field.name}"] = field.value

                    all_normalized_data.append(request_data)

                page += 1

            if all_normalized_data:
                df = pd.DataFrame(all_normalized_data)

                # Replace '/' in column names
                df.columns = [col.replace("/", "_") for col in df.columns]

                # **Remove whitespace from specific columns:**
                columns_to_edit = ["response_data_DateSigned"]
                for col in columns_to_edit:
                    if col in df.columns:
                        df[col] = df[col].str.replace(' ','')

            if 'response_data_DateSigned' in df.columns:
                  df['response_data_DateSigned'] = pd.to_datetime(df['response_data_DateSigned'], errors='coerce')
                  print(f"Data type of 'signature_1_signed_at' after conversion: {df['response_data_DateSigned'].dtype}")

                  #csv_filename = "all_signature_requests.csv"
                  #df.to_csv(csv_filename, index=False)
                  #print(f"All data exported to {csv_filename}")
                  return df

            else:
                print("No signature requests found.")
                return None

        except ApiException as e:
            print(f"Exception when calling Dropbox Sign API: {e}")
            return None
        except Exception as e:
            print(f"An unexpected error occurred: {e}")
            return None


# Example Usage
api_key = ds_api  
account_id = ds_acct
table_id = "signatures.Dropbox_Sign"
project_id = "prod"

result_df = normalize_all_signature_requests(api_key, account_id)
if result_df is not None:
    print(result_df)

In [5]:
import pandas as pd

result_df.rename(columns={'response_data_Address': 'data_Address'}, inplace=True)

In [ ]:
import pandas_gbq
table_id = "amyhs.Sign"
project_id = "prod"
dataset_id = "prod.amyhs"
pandas_gbq.to_gbq(result_df, table_id, project_id=project_id, if_exists='replace')